To run this fenic demo, click **Runtime** > **Run all**.

<div class="align-center">
<a href="https://github.com/typedef-ai/fenic"><img src="https://github.com/typedef-ai/fenic/blob/main/docs/images/typedef-fenic-logo-github-yellow.png?raw=true" height="50"></a>
<a href="https://discord.gg/GdqF3J7huR"><img src="https://github.com/typedef-ai/fenic/blob/main/docs/images/join-the-discord.png?raw=true" height="50"></a>
<a href="https://docs.fenic.ai/latest/"><img src="https://github.com/typedef-ai/fenic/blob/main/docs/images/documentation.png?raw=true" height="50"></a>

Questions? Join the Discord and ask away! For feature requests or to leave a star, visit our [GitHub](https://github.com/typedef-ai/fenic).

</div>


# 📄 fenic Demo: PDF Processing, Analysis, and Content Discovery


Research papers, whitepapers, technical documents - PDFs contain valuable information but are notoriously difficult to work with. Traditional PDF processing requires complex parsing, layout analysis, and manual extraction. 

In this demo, we'll build an end-to-end pipeline for parsing PDF content, adding structure, deduping, then create MCP tools to easily query the clean, structured dataset you've created.

**What you'll see in this demo:**
- 📚 **PDF to Markdown**: Intelligent conversion preserving document structure and formatting
- 🕋 **Chunking and Deduplication**: Use fenic's text manipulation and fuzzy matching algorithms to dedup structured data
- 🧠 **Content Categorization**: Automatic classification of document sections
- 📊 **Structured Extraction**: Products, training methods, key topics identified
- ⚡ **Batch Processing**: Multiple PDFs processed and analyzed efficiently
- 🛰️ **MCP Server:** MCP server that exposes your curated whitepaper dataset as tools for MCP clients and agents


Perfect for research analysis, document management, and content discovery.

## 🔩 Setup and Installs

Use pinned installs required by the demo. The wheel below includes a prompt fix that improves Markdown header parsing. If you don’t need the patch, use the safer alternative right after this cell.

In [ ]:
!pip uninstall -y sklearn-compat ibis-framework imbalanced-learn google-genai
!pip install polars==1.30.0
!pip install huggingface_hub
!pip install fenic
!pip install google-genai[local-tokenizer]>=1.36.0

# === GOOGLE GEMINI ===
#!pip install fenic[google]
# === ANTHROPIC CLAUDE ===
#!pip install fenic[anthropic]
# === OPENAI (Default) ===
#!pip install "fenic[google]

# Installs for the MCP server section
!pip -q install "fenic[mcp]" uvicorn fastmcp websockets starlette

Found existing installation: ibis-framework 9.5.0
Uninstalling ibis-framework-9.5.0:
  Successfully uninstalled ibis-framework-9.5.0
Found existing installation: imbalanced-learn 0.14.0
Uninstalling imbalanced-learn-0.14.0:
  Successfully uninstalled imbalanced-learn-0.14.0
Found existing installation: google-genai 1.54.0
Uninstalling google-genai-1.54.0:
  Successfully uninstalled google-genai-1.54.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 25.8 MB/s eta 0:00:00
  Attempting uninstall: polars
    Found existing installation: polars 1.31.0
    Uninstalling polars-1.31.0:
      Successfully uninstalled polars-1.31.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.5/14.5 MB 114.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

🔌 MULTI-PROVIDER SETUP - Choose your preferred LLM provider

Uncomment provider sections you want to use later in your semantic config

In [ ]:
import os, shutil, getpass, pathlib

# 🔌 MULTI-PROVIDER SETUP - Choose your preferred LLM provider
# Uncomment provider sections you are using in your semantic config

# === OPENAI (Default) ===
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

# === GOOGLE GEMINI ===
os.environ["GOOGLE_API_KEY"] = getpass.getpass("Google API Key:")

# === ANTHROPIC CLAUDE ===
# os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API Key:")

OpenAI API Key:··········
Google API Key:··········


## 🪛 Step 1: Imports and Configs

In [ ]:
# === IMPORTS + small additions for robustness ===

from typing import List, Dict, Any, Optional

import huggingface_hub as hf
from huggingface_hub import list_repo_files, hf_hub_download
from pydantic import BaseModel, Field

import fenic as fc
from fenic.api.functions import semantic, text, embedding

# --- Optional: honor pre-set directories from earlier cells; else set sensible defaults ---
DATA_DIR = os.environ.get("DATA_DIR", "/content/sample_pdfs")
OUT_DIR = os.environ.get("OUT_DIR", "/content/out")
PREVIEW_DIR = f"{OUT_DIR}/preview"

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(PREVIEW_DIR, exist_ok=True)

# --- Provider keys (read from env) ---
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

if not (OPENAI_API_KEY or GOOGLE_API_KEY):
    print("⚠️  No provider keys detected. Set OPENAI_API_KEY and/or GOOGLE_API_KEY before extraction steps.")

Keep your two-model setup:
- `parse_model`: usually a more permissive/structured parser (Gemini here)
- `semantic_model`: default for semantic ops (OpenAI here)

In [ ]:
# --- Multi-provider semantic configuration ---
session = fc.Session.get_or_create(
    fc.SessionConfig(
        app_name="pdf_process_and_dedup_demo2",
        semantic=fc.SemanticConfig(
            language_models={
                "parse_model": fc.GoogleDeveloperLanguageModel(
                    model_name="gemini-2.5-pro",
                    rpm=500,
                    tpm=1_000_000,
                    api_key=GOOGLE_API_KEY,   # uses env var if not provided
                ),
                "semantic_model": fc.OpenAILanguageModel(
                    model_name="gpt-4o-mini",
                    rpm=500,
                    tpm=200_000,
                    api_key=OPENAI_API_KEY,   # uses env var if not provided
                ),
            },
            embedding_models={
                "text-embedding-3-small": fc.OpenAIEmbeddingModel(
                model_name="text-embedding-3-small", rpm=3000, tpm=5_000_000
            )
            },
            default_language_model="semantic_model",
            default_embedding_model="text-embedding-3-small"
        ),
    )
)

print("✅ Session configured:")
print("   • Parser LLM   : Google Gemini 2.5 Pro (parse_model)")
print("   • Semantic LLM : OpenAI gpt-4o-mini (semantic_model)")
print(f"   • OUT_DIR      : {OUT_DIR}")
print(f"   • DATA_DIR     : {DATA_DIR}")

✅ Session configured:
   • Parser LLM   : Google Gemini 2.5 Pro (parse_model)
   • Semantic LLM : OpenAI gpt-4o-mini (semantic_model)
   • OUT_DIR      : /content/out
   • DATA_DIR     : /content/sample_pdfs


## 📑 Step 1: Download PDFs Dataset from Hugging Face


Let's grab some real whitepapers to process - these are complex technical documents perfect for demonstrating AI-powered PDF analysis.

This dataset includes duplicate PDFs, and a PDF with some duplicate pages, so we can demonstrate a few ways fenic can be used to deduplicate content.

**Source Repo:** `typedef-ai/pdf_data` (HF dataset)


In [ ]:
# 📚 Download sample whitepapers from Hugging Face

REPO_ID = "typedef-ai/pdf_data"
files = hf.list_repo_files(repo_id=REPO_ID, repo_type="dataset")

print(f"📥 Downloading whitepapers from {REPO_ID}...")
for file in files:
    if file.startswith("whitepapers/dedup"):
        hf.hf_hub_download(repo_id=REPO_ID, repo_type="dataset", filename=file, local_dir=DATA_DIR)
        print(f"  ✅ Downloaded: {file}")

print(f"📁 PDFs saved to: {DATA_DIR}")

📥 Downloading whitepapers from typedef-ai/pdf_data...


.DS_Store:   0%|          | 0.00/6.15k [00:00<?, ?B/s]

  ✅ Downloaded: whitepapers/dedup/.DS_Store


whitepapers/dedup/PA_whitepaper_10-most-(…):   0%|          | 0.00/740k [00:00<?, ?B/s]

  ✅ Downloaded: whitepapers/dedup/PA_whitepaper_10-most-critical-security-risks-in-serverless-architectures-guide.pdf


...

  ✅ Downloaded: whitepapers/dedup/PA_whitepaper_ai-governance_copy_1.pdf


whitepapers/dedup/PA_whitepaper_ai-gover(…):   0%|          | 0.00/461k [00:00<?, ?B/s]

  ✅ Downloaded: whitepapers/dedup/PA_whitepaper_ai-governance_copy_2.pdf


whitepapers/dedup/PA_whitepaper_securing(…):   0%|          | 0.00/244k [00:00<?, ?B/s]

  ✅ Downloaded: whitepapers/dedup/PA_whitepaper_securing-the-soc-with-nec-xon.pdf
📁 PDFs saved to: /content/sample_pdfs


## ✅ Step 2: Filter Dataset + Preview Table

### **What this cell does:**
- Read PDF metadata with **`session.read.pdf_metadata()`**  
- Keep just the columns we need and **order** deterministically  
- **Preview** rows with `.show()`  
- **Persist** a reproducible CSV to `OUT_DIR/preview/files.csv` via `df.write.csv()`  
- Compute and print **total pages across all PDFs** (fenic aggregation)

In [ ]:
# Use recursive glob to match nested paths like .../whitepapers/dedup/*.pdf
PDF_GLOB = f"{DATA_DIR}/**/*.pdf"

# 1) Read PDF metadata via Fenic (recursive)
pdf_df = session.read.pdf_metadata(PDF_GLOB, recursive=True)

# 2) Show available columns + row count
print("🧬 Columns:", [f.name for f in pdf_df.schema.column_fields])
print("📦 File count:", pdf_df.count())

🧬 Columns: ['file_path', 'error', 'size', 'title', 'author', 'creation_date', 'mod_date', 'page_count', 'has_forms', 'has_signature_fields', 'image_count', 'is_encrypted']
📦 File count: 5


In [ ]:
# 3) Build compact preview (stable names)
preview_df = (
    pdf_df
    .select(
        fc.col("file_path").alias("file_path"),
        fc.col("page_count").alias("n_pages"),
        fc.col("size").alias("size_bytes"),
        fc.col("title").alias("title"),
        fc.col("author").alias("author"),
    )
    .order_by(fc.col("file_path"))
)

print("🔎 Preview of detected PDFs:")
preview_df.show(10)

🔎 Preview of detected PDFs:
┌────────────────────────────┬─────────┬────────────┬────────────────────────────┬─────────────────┐
│ file_path                  ┆ n_pages ┆ size_bytes ┆ title                      ┆ author          │
╞════════════════════════════╪═════════╪════════════╪════════════════════════════╪═════════════════╡
│ /content/sample_pdfs/white ┆ 24      ┆ 740017     ┆ Copy of PA_whitepaper_10-m ┆                 │
│ papers/dedup/PA_whitepaper ┆         ┆            ┆ ost-critical-security-risk ┆                 │
│ _10-most-critical-security ┆         ┆            ┆ s-in-serverless-architectu ┆                 │
│ -risks-in-serverless-archi ┆         ┆            ┆ res-guide                  ┆                 │
│ tectures-guide.pdf         ┆         ┆            ┆                            ┆                 │
│ /content/sample_pdfs/white ┆ 14      ┆ 460824     ┆ AI Governance for          ┆                 │
│ papers/dedup/PA_whitepaper ┆         ┆            ┆ AI-Powere

In [ ]:
# 4) Total pages across all PDFs
total_pages_df = (
    pdf_df
    .select(fc.col("page_count"))
    .group_by()  # global aggregation
    .agg(fc.sum(fc.col("page_count")).alias("total_pages"))
)
print("🧮 Total pages across all PDFs:")
total_pages_df.show()

🧮 Total pages across all PDFs:
┌─────────────┐
│ total_pages │
╞═════════════╡
│ 72          │
└─────────────┘


## 🧠 Step 3: AI-Powered PDF to Markdown Conversion


Now the magic happens! Watch AI convert complex PDFs into clean, structured markdown while preserving all the important formatting and hierarchy.

First we can filter which documents we parse based on the PDF metadata.  In this case, we're only interested in longer, unencrypted documents.

In [ ]:
pdf_filtered_df = session.read.pdf_metadata(f"{DATA_DIR}/**/*.pdf", recursive=True).filter(
    (fc.col("page_count") > 3) & (~fc.col("is_encrypted"))
)

print(f"📊 Found {pdf_filtered_df.count()} valid PDFs to process")
pdf_filtered_df.select("title", "page_count", "file_path").show()

📊 Found 5 valid PDFs to process
┌──────────────────────────────────────────┬────────────┬──────────────────────────────────────────┐
│ title                                    ┆ page_count ┆ file_path                                │
╞══════════════════════════════════════════╪════════════╪══════════════════════════════════════════╡
│ AI Governance for AI-Powered             ┆ 14         ┆ /content/sample_pdfs/whitepapers/dedup/P │
│ Applications                             ┆            ┆ A_whitepaper_ai-governance_copy_2.pdf    │
│ AI Governance for AI-Powered             ┆ 14         ┆ /content/sample_pdfs/whitepapers/dedup/P │
│ Applications                             ┆            ┆ A_whitepaper_ai-governance_copy_0.pdf    │
│ Securing the Modern SOC with NEC XON and ┆ 6          ┆ /content/sample_pdfs/whitepapers/dedup/P │
│ Palo Alto Networks                       ┆            ┆ A_whitepaper_securing-the-soc-with-nec-x │
│                                          ┆            ┆ o

Next, we'll batch the PDF pages to a VLM and get back Markdown content.

We'll add `page_separator` because we want to chunk the content by page in later sections of the demo!


In [ ]:
# 🚀 Convert PDFs to Markdown using AI
print("🤖 Converting PDFs to markdown using Gemini...")
pdf_to_md_content = (
    pdf_filtered_df.with_column(
    "markdown_content",
    fc.semantic.parse_pdf(fc.col("file_path"), model_alias="parse_model", page_separator="--- PAGE BREAK ---")
).cache()
)

print("✅ PDF to Markdown conversion complete!")
print(f"\n 📄 Processed {pdf_to_md_content.count()} documents")

# Quick sanity: show paths; markdown will be used in the next step
pdf_to_md_content.select("file_path", "page_count").order_by("file_path").show(10)

🤖 Converting PDFs to markdown using Gemini...
✅ PDF to Markdown conversion complete!


Submitting requests for batch: f729fd60-7e80-455e-b094-f7ea31c9e312 (model: gemini-2.5-pro): 100%|██████████| 25/25 [00:00<00:00, 32.89req/s, estimated_input_tokens=18576, estimated_output_tokens=77153]
Awaiting responses for batch f729fd60-7e80-455e-b094-f7ea31c9e312 (model: gemini-2.5-pro): 100%|██████████| 25/25 [00:27<00:00,  1.10s/res]


 📄 Processed 5 documents
┌─────────────────────────────────────────────────────────────────────────────────────┬────────────┐
│ file_path                                                                           ┆ page_count │
╞═════════════════════════════════════════════════════════════════════════════════════╪════════════╡
│ /content/sample_pdfs/whitepapers/dedup/PA_whitepaper_10-most-critical-security-risk ┆ 24         │
│ s-in-serverless-architectures-guide.pdf                                             ┆            │
│ /content/sample_pdfs/whitepapers/dedup/PA_whitepaper_ai-governance_copy_0.pdf       ┆ 14         │
│ /content/sample_pdfs/whitepapers/dedup/PA_whitepaper_ai-governance_copy_1.pdf       ┆ 14         │
│ /content/sample_pdfs/whitepapers/dedup/PA_whitepaper_ai-governance_copy_2.pdf       ┆ 14         │
│ /content/sample_pdfs/whitepapers/dedup/PA_whitepaper_securing-the-soc-with-nec-xon. ┆ 6          │
│ pdf                                                            

## 📊 Step 4: Extract Document Structure and Table of Contents



Now that we have Markdown for each PDF, we’ll derive a **document structure** that’s easy to navigate and query.

**What this cell does**
- Builds a human-friendly **name** per document (PDF title if present, otherwise the filename/path).
- Generates a **Table of Contents (TOC)** from Markdown headings using fenic’s Markdown utilities.
- Retains the original `file_path`, `page_count`, and `markdown_content` for downstream steps.

**Why this matters**
- A TOC gives quick visibility into section hierarchy (H1–H3) and improves targeted extraction later.
- Consistent naming helps align artifacts (Markdown files, chunks, schema rows).

**Output columns**
- `name` – display name for the document
- `file_path` – absolute path on disk
- `page_count` – number of pages
- `markdown_content` – full Markdown text for the document
- `toc` – Markdown-formatted table of contents
- `sections`



Continue from `pdf_to_md_content` produced in Step 3.


- Columns available: `file_path`, `page_count`, `title`, `markdown_content`, …

In [ ]:
# 1) Build a display name: prefer PDF title

pdf_md_content = pdf_to_md_content.select(
    fc.when(
        fc.col("title").is_not_null(),
        fc.col("title")
    ).otherwise(
        fc.text.split_part(fc.col("file_path"), "/", -1)
    ).alias("name"),
    fc.col("markdown_content"),
    # 2) Generate a Markdown Table of Contents from headings
    fc.markdown.generate_toc(fc.col("markdown_content")).alias("toc"),
    "page_count",
)

pdf_md_content_indexed = session.create_dataframe(
    pdf_md_content.to_polars().with_row_index("id")
)
# annoying because Polars doesn't support markdown type, and we don't have a with_row_index method yet.
pdf_md_content_indexed = pdf_md_content_indexed.with_column(
    "markdown_content", fc.col("markdown_content").cast(fc.MarkdownType)
).with_column(
    "toc", fc.col("toc").cast(fc.MarkdownType)
)

print("📊 Document structure extracted.")

📊 Document structure extracted.


In [ ]:
# (Optional) quick peek at a single TOC to verify heading extraction
pdf_md_content_indexed.select("name", "toc", "markdown_content").order_by("name").show(1)

┌────────────────────────────────┬────────────────────────────────┬────────────────────────────────┐
│ name                           ┆ toc                            ┆ markdown_content               │
╞════════════════════════════════╪════════════════════════════════╪════════════════════════════════╡
│ AI Governance for AI-Powered   ┆ # Establishing a Governance    ┆ # Establishing a Governance    │
│ Applications                   ┆ Framework for Al-Powered       ┆ Framework for Al-Powered       │
│                                ┆ Applications                   ┆ Applications                   │
│                                ┆ ## Current Al Landscape and    ┆                                │
│                                ┆ Its Security Implications      ┆ **Artificial intelligence      │
│                                ┆ ### "Traditional" AI and       ┆ (AI)** is advancing rapidly,   │
│                                ┆ Machine Learning               ┆ with organizations acro

In [ ]:
# ✅ Robust TOC emptiness check (handles NULLs + whitespace)
empty_tocs = pdf_md_content_indexed.filter(
    fc.col("toc").is_null() |
    (fc.text.length(fc.text.trim(fc.col("toc").cast(fc.StringType))) == 0)
)

print("🧪 Docs with empty TOC:", empty_tocs.count())
empty_tocs.select("name").show()

🧪 Docs with empty TOC: 0
┌──────┐
│ name │
╞══════╡
└──────┘


## 🧹 Step 5: Deduplicate Documents (TOC Text Similarity)

fenic can dedup the structured text and remove near-duplicate PDFs cheaply **without LLMs** by comparing their **Table of Contents (TOC)** strings using a fast Levenshtein fuzzy ratio. This keeps the dataset lean before page-level work.

In the previous step we extracted a Table of Contents from the markdown.  Rather than comparing the whole text, let's eliminate any documents that have similar header structures.

We'll take the toc (Table of Contents) processed from the PDF's headers, perform fuzzy matching, and eliminate any close matches.


**What this cell does**

* Creates a stable `doc_id` from `file_path` and lists candidate docs.
* Builds **all unique pairs** of documents (no self-pairs, no A–B vs B–A duplicates).
* Computes **TOC similarity** via `compute_fuzzy_ratio(..., method="levenshtein")`.
* Drops documents whose TOC is **≥ threshold** similar to another (defaults to 90).
* Carries forward the **deduped set** for downstream chunking and extraction.

**Why this matters**

* Dramatically reduces wasted compute on duplicates (common in downloaded corpora).
* TOC-level matching is cheap and robust to small content changes while catching true duplicates.



In [ ]:
# Check to see if there are duplicates from the titles:

pdf_md_content_indexed.select("id", "name").show()

┌────┬─────────────────────────────────────────────────────────────────────────────────────────┐
│ id ┆ name                                                                                    │
╞════╪═════════════════════════════════════════════════════════════════════════════════════════╡
│ 0  ┆ AI Governance for AI-Powered Applications                                               │
│ 1  ┆ AI Governance for AI-Powered Applications                                               │
│ 2  ┆ Securing the Modern SOC with NEC XON and Palo Alto Networks                             │
│ 3  ┆ Copy of PA_whitepaper_10-most-critical-security-risks-in-serverless-architectures-guide │
│ 4  ┆ AI Governance for AI-Powered Applications                                               │
└────┴─────────────────────────────────────────────────────────────────────────────────────────┘


Cross join the table of contents (TOC) and `id`:

In [ ]:
pdf_toc_df = pdf_md_content_indexed.select("id","name", "toc")
pdf_toc_comp_df = pdf_toc_df.select(fc.col("id").alias("id_comp"),fc.col("toc").alias("toc_comp"), fc.col("name").alias("name_comp"))

Calculate fuzzy ratio between all combinations of TOC contents:

In [ ]:
result = pdf_toc_df.join(
    pdf_toc_comp_df, how="cross"
).filter(
    fc.col("id") < fc.col("id_comp") # avoid duplicates and self-comparisons
).with_column(
    "fuzzy_ratio",
    fc.text.compute_fuzzy_ratio(fc.col("toc").cast(fc.StringType), fc.col("toc_comp").cast(fc.StringType), method="levenshtein").alias("fuzzy_ratio")
)

Identify ids of documents with very similar TOC contents.


We'll eliminate anything with a fuzzy similarity score greater than 90%

In [ ]:
dedupe_ids = result.filter(fc.col("fuzzy_ratio") > 90).select(fc.col("id_comp").alias("id")).drop_duplicates()
print("-"*75)
print(f"Found {dedupe_ids.count()} duplicate documents:")
print("-"*75)

result.filter(fc.col("fuzzy_ratio") > 90).select("id", "id_comp", "fuzzy_ratio").show()#, "toc", "toc_comp").show()
dedupe_ids = dedupe_ids.agg(fc.collect_list("id").alias("ids"))

---------------------------------------------------------------------------
Found 2 duplicate documents:
---------------------------------------------------------------------------
┌────┬─────────┬─────────────┐
│ id ┆ id_comp ┆ fuzzy_ratio │
╞════╪═════════╪═════════════╡
│ 0  ┆ 1       ┆ 95.674831   │
│ 0  ┆ 4       ┆ 96.55914    │
│ 1  ┆ 4       ┆ 96.404377   │
└────┴─────────┴─────────────┘


Drop the identified documents


In [ ]:
deduped_docs_pdf_content = pdf_md_content_indexed.join(dedupe_ids, how="cross").filter(~fc.array_contains(fc.col("ids"), fc.col("id"))).drop("ids")

print("-"*75)
print(f"Documents remaining after deduplication:")
print("-"*75)
deduped_docs_pdf_content.select("name", "id").show()

---------------------------------------------------------------------------
Documents remaining after deduplication:
---------------------------------------------------------------------------
┌─────────────────────────────────────────────────────────────────────────────────────────┬────┐
│ name                                                                                    ┆ id │
╞═════════════════════════════════════════════════════════════════════════════════════════╪════╡
│ AI Governance for AI-Powered Applications                                               ┆ 0  │
│ Securing the Modern SOC with NEC XON and Palo Alto Networks                             ┆ 2  │
│ Copy of PA_whitepaper_10-most-critical-security-risks-in-serverless-architectures-guide ┆ 3  │
└─────────────────────────────────────────────────────────────────────────────────────────┴────┘


### 📃 Step 5.1: Deduplicate at Page Level



Let's show how fenic can find duplicates in PDF pages after the parse step.After converting PDFs to Markdown, we’ll remove **near-duplicate pages** inside each document so downstream extraction isn’t polluted by repeated content (e.g., duplicated slides, boilerplate pages).

**What this cell does**

* Splits each document into **page rows** using the parse-time separator (`--- PAGE BREAK ---`).
* Uses a UDF to **strip Markdown** → raw text for fair string comparisons.
* Builds candidate pairs **within the same document** (no cross-doc pairs).
* **Blocks** pairs by short hash bucket and **length band** to avoid O(n²) work.
* Applies **Levenshtein** fuzzy ratio and drops the lower-priority page in each near-duplicate pair.

**Why this matters**

* Cuts redundant compute for chunking, classification, and schema extraction.
* Improves accuracy of summaries and table extraction by removing repeated pages.

**Output columns**

* `doc_id` – document identifier (from `file_path`)
* `name` – display name for the document
* `page_id` – stable hash of the raw page text
* `raw_page_text` – Markdown-stripped content for that page

**Tunable knobs**

* `THRESHOLD` (default **85**) – similarity cutoff for page duplicates.
* **Blocking:** short hash prefix length (default 6) and ±**10%** length band.

**Quick sanity checks**

* Page count after dedup < before, especially for docs with obvious repeats.
* High-score candidate pairs printed look like true duplicates (title pages, legal notices, etc.).


In [ ]:
# get the raw text, remove all markdown structure for content comparison
from markdown_it import MarkdownIt
def _extract_markdown_text(content: str) -> str:
    md = MarkdownIt()
    tokens = md.parse(content)

    # Flatten all inline text content
    texts = []
    for t in tokens:
        if t.type == "inline":
            texts.extend([c.content for c in t.children if c.type == "text"])

    return " ".join(texts)

udf_extract_raw_md_text = fc.udf(
    _extract_markdown_text,
    return_type=fc.StringType
)

Chunk by pages:

In [ ]:
# use the page break delimiter we provided during the parse step
pdf_page_lists = (
    deduped_docs_pdf_content.select("id", "name", "page_count", fc.text.split(fc.col("markdown_content")
    .cast(fc.StringType), "--- PAGE BREAK ---")
    .alias("page_text"))
    )

all_pdf_pages = pdf_page_lists.explode("page_text").with_column("doc_id", fc.col("id")).drop("id")

# Add per page id
all_pdf_pages = (
    session.create_dataframe(all_pdf_pages.to_polars()
    .with_row_index("page_id"))
    .cache()
    )

all_pdf_pages.show()

┌─────────┬─────────────────────────────────┬────────────┬────────────────────────────────┬────────┐
│ page_id ┆ name                            ┆ page_count ┆ page_text                      ┆ doc_id │
╞═════════╪═════════════════════════════════╪════════════╪════════════════════════════════╪════════╡
│ 0       ┆ AI Governance for AI-Powered    ┆ 14         ┆ # Establishing a Governance    ┆ 0      │
│         ┆ Applications                    ┆            ┆ Framework for Al-Powered       ┆        │
│         ┆                                 ┆            ┆ Applications                   ┆        │
│         ┆                                 ┆            ┆                                ┆        │
│         ┆                                 ┆            ┆ **Artificial intelligence      ┆        │
│         ┆                                 ┆            ┆ (AI)** is advancing rapidly,   ┆        │
│         ┆                                 ┆            ┆ with organizations across many ┆

In [ ]:
raw_page_text_df = all_pdf_pages.select("name", "doc_id", "page_id", udf_extract_raw_md_text(fc.col("page_text")).alias("raw_page_text")).cache()

dup_page_ids = []

# We only want to deduplicate pages in the context of the same document.
for doc_dict in raw_page_text_df.select("doc_id", "name").drop_duplicates().to_pylist():
    # Grab the page ids and raw page content
    dedup_pages = raw_page_text_df.filter((fc.col("doc_id") == doc_dict["doc_id"])).select("page_id", "raw_page_text")
    dedup_pages_comp = dedup_pages.select(fc.col("page_id").alias("page_id_comp"), fc.col("raw_page_text").alias("raw_page_text_comp"))

    # Use fenic's RapidFuzz matching to find duplicate pages
    result = dedup_pages.join(
        dedup_pages_comp, how="cross"
    ).filter(
        fc.col("page_id") < fc.col("page_id_comp")
    ).with_column(
        "fuzzy_ratio_levenshtein",
        fc.text.compute_fuzzy_ratio(
            fc.col("raw_page_text"), fc.col("raw_page_text_comp").cast(fc.StringType), method="levenshtein"
        )
    ).with_column(
        "fuzzy_ratio_jaro",
        fc.text.compute_fuzzy_ratio(
            fc.col("raw_page_text"), fc.col("raw_page_text_comp").cast(fc.StringType), method="jaro"
        )
    ).with_column(
        "fuzzy_ratio_jaro_winkler",
        fc.text.compute_fuzzy_ratio(
            fc.col("raw_page_text"), fc.col("raw_page_text_comp").cast(fc.StringType), method="jaro_winkler"
        )
    ).with_column(
        "fuzzy_ratio_hamming",
        fc.text.compute_fuzzy_ratio(
            fc.col("raw_page_text"), fc.col("raw_page_text_comp").cast(fc.StringType), method="hamming"
        )
    ).with_column(
        "fuzzy_ratio",
        fc.greatest(fc.col("fuzzy_ratio_levenshtein"), fc.col("fuzzy_ratio_jaro"), fc.col("fuzzy_ratio_jaro_winkler"), fc.col("fuzzy_ratio_hamming"))
    )

    #result.filter(fc.col("fuzzy_ratio") > 85).select("fuzzy_ratio_levenshtein", "fuzzy_ratio_jaro", "fuzzy_ratio_jaro_winkler", "fuzzy_ratio_hamming", "fuzzy_ratio", "page_id", "page_id_comp").show()

    dedup_ids = result.filter(fc.col("fuzzy_ratio") > 85).select(fc.col("page_id_comp").alias("page_id")).drop_duplicates()
    print("="*70)
    print(f"Comparing {result.count()} page combinations for document '{doc_dict['name']}'...\n\n")

    print(f"Found {dedup_ids.count()} duplicate pages.")
    print("="*70+"\n")

    # track the duplicate page ids for each document so we can prune later
    dedup_ids = dedup_ids.agg(fc.collect_list("page_id").alias("dup_page_ids"))
    if dedup_ids.count() > 0:
        dup_page_ids.extend(dedup_ids.to_pylist()[0]["dup_page_ids"])

Comparing 15 page combinations for document 'Securing the Modern SOC with NEC XON and Palo Alto Networks'...


Found 0 duplicate pages.

Comparing 210 page combinations for document 'Copy of PA_whitepaper_10-most-critical-security-risks-in-serverless-architectures-guide'...


Found 2 duplicate pages.

Comparing 105 page combinations for document 'AI Governance for AI-Powered Applications'...


Found 0 duplicate pages.



In [ ]:
# (Optional) tiny sanity peek helpers (non-disruptive)

print("Total dup page_ids collected:", len(dup_page_ids))
# If you want to see which pages were flagged (by doc):
raw_page_text_df.filter(fc.array_contains(fc.lit(dup_page_ids), fc.col("page_id")))\
    .select("name","doc_id","page_id")\
    .order_by(["name","page_id"])\
    .show(20)

Total dup page_ids collected: 2
┌───────────────────────────────────────────────────────────────────────────────┬────────┬─────────┐
│ name                                                                          ┆ doc_id ┆ page_id │
╞═══════════════════════════════════════════════════════════════════════════════╪════════╪═════════╡
│ Copy of PA_whitepaper_10-most-critical-security-risks-in-serverless-architect ┆ 3      ┆ 33      │
│ ures-guide                                                                    ┆        ┆         │
│ Copy of PA_whitepaper_10-most-critical-security-risks-in-serverless-architect ┆ 3      ┆ 37      │
│ ures-guide                                                                    ┆        ┆         │
└───────────────────────────────────────────────────────────────────────────────┴────────┴─────────┘


### 🧩 Step 5.2: Rebuild Markdown After Page-Level Dedup

Now we’ll **map the kept page keys back to their original Markdown text** and reconstruct a clean per-document `markdown_content`. This preserves headings, lists, and formatting while removing duplicate pages.

**What this cell does**

* Joins **kept pages** (`doc_id`, `page_key`) to the original **page Markdown**.
* Reassembles each document with the same `PAGE_SEP` used during parsing.
* Updates the working content frame with **deduped Markdown**.

**Inputs**

* `deduped_docs_pdf_content` (`doc_id`, `name`, `page_key`, `raw_page_text`)
* `pdf_page_lists` from Step 5.2 (list of page Markdown per doc)
* `deduped_docs_pdf_content` (pre-dedup content frame)


Add the duplicate page ids to the dataframe:

In [ ]:

all_pdf_pages_dedup = all_pdf_pages.with_column(
    "dup_page_ids",
    fc.lit(dup_page_ids)
).filter(~fc.array_contains(fc.col("dup_page_ids"), fc.col("page_id")))

# Sanity check the deduplication
print (f"Total Pages before deduplication: {all_pdf_pages.count()}")
print (f"Total Pages after deduplication: {all_pdf_pages_dedup.count()}")

# Reform the markdown content per PDF doc
deduped_markdown_content = all_pdf_pages_dedup.group_by("doc_id").agg(
    fc.collect_list(fc.col("page_text")).alias("page_text_array")
).select(
    "doc_id",
    fc.text.array_join(fc.col("page_text_array"), "--- PAGE BREAK ---").cast(fc.MarkdownType).alias("markdown_content")
)

# Update our content dataframe with the deduped markdown content
deduped_docs_pdf_content_final = deduped_docs_pdf_content.with_column(
    'doc_id', fc.col('id')
).drop('markdown_content'
).join(deduped_markdown_content, how="left", on="doc_id").drop('doc_id').cache()

Total Pages before deduplication: 42
Total Pages after deduplication: 40


Per-doc page counts after dedup (based on separator):

In [ ]:
PAGE_SEP = "--- PAGE BREAK ---"

dedup_page_counts = (
    deduped_docs_pdf_content_final
      .select(
          "name",
          fc.text.split(fc.col("markdown_content").cast(fc.StringType), PAGE_SEP).alias("pages")
      )
      .explode("pages")
      .group_by("name")
      .agg(fc.count(fc.col("pages")).alias("pages_after_dedup"))
      .order_by("name")
)
dedup_page_counts.show()


# Ensure no empty markdown slipped in
empty_md_df = deduped_docs_pdf_content_final.filter(
    fc.col("markdown_content").is_null()
    | (fc.col("markdown_content").cast(fc.StringType) == "")
)
print("Empty/Null markdown docs:", empty_md_df.count())
#empty_md_df.select("name").show()

┌──────────────────────────────────────────────────────────────────────────────┬───────────────────┐
│ name                                                                         ┆ pages_after_dedup │
╞══════════════════════════════════════════════════════════════════════════════╪═══════════════════╡
│ AI Governance for AI-Powered Applications                                    ┆ 15                │
│ Copy of PA_whitepaper_10-most-critical-security-risks-in-serverless-architec ┆ 19                │
│ tures-guide                                                                  ┆                   │
│ Securing the Modern SOC with NEC XON and Palo Alto Networks                  ┆ 6                 │
└──────────────────────────────────────────────────────────────────────────────┴───────────────────┘
Empty/Null markdown docs: 0


## 🧠 Step 6: AI-Powered Content Analysis

Now that we've done some deduplication with cheap in memory libraries, lets use LLMs to extract semantic information and classifications from the content!


### 📜 Step 6.1: Structure Content Around the Header Sections

First, lets add more structure to our PDF content using fenic's markdown support.  In this case, lets divide the deduped PDFs into sections by header.

Extract sections up to level-3 headers (H1–H3), attach them, and sanity-check counts:

In [ ]:
# 1) Extract header chunks at each level
pdf_sections_1 = (
    deduped_docs_pdf_content_final
      .select("id", fc.markdown.extract_header_chunks(fc.col("markdown_content"), header_level=1).alias("sections"))
      .explode("sections")
)

pdf_sections_2 = (
    deduped_docs_pdf_content_final
      .select("id", fc.markdown.extract_header_chunks(fc.col("markdown_content"), header_level=2).alias("sections"))
      .explode("sections")
)

pdf_sections_3 = (
    deduped_docs_pdf_content_final
      .select("id", fc.markdown.extract_header_chunks(fc.col("markdown_content"), header_level=3).alias("sections"))
      .explode("sections")
)

# 2) Combine into one array-of-sections per doc
pdf_sections_df = (
    pdf_sections_1
      .union(pdf_sections_2)
      .union(pdf_sections_3)
      .group_by("id")
      .agg(fc.collect_list("sections").alias("sections"))
)

# 3) Join sections back to the main table
pdf_md_content_indexed = deduped_docs_pdf_content_final.join(pdf_sections_df, on="id")

In [ ]:
# 4) Section counts per whitepaper
section_counts = (
    pdf_md_content_indexed
      .select(
          "id",
          "name",
          fc.coalesce(fc.arr.size(fc.col("sections")), fc.lit(0)).alias("n_sections")
      )
      .order_by(fc.col("n_sections").desc())
)

print("\n🧾 Sections per whitepaper:")
section_counts.show()

# 5) Optional: breakdown by header level (across all docs)
level_counts = (
    pdf_md_content_indexed
      .explode("sections")
      .select(fc.col("sections").level.alias("level"))
      .group_by("level")
      .agg(fc.count(fc.lit(1)).alias("n_sections"))
      .order_by("level")
)

print("\n📊 Sections by header level:")
level_counts.show()

print("\n✅ Done! `pdf_md_content_indexed` now includes a populated `sections` column.")


🧾 Sections per whitepaper:
┌────┬────────────────────────────────────────────────────────────────────────────────┬────────────┐
│ id ┆ name                                                                           ┆ n_sections │
╞════╪════════════════════════════════════════════════════════════════════════════════╪════════════╡
│ 3  ┆ Copy of PA_whitepaper_10-most-critical-security-risks-in-serverless-architectu ┆ 38         │
│    ┆ res-guide                                                                      ┆            │
│ 0  ┆ AI Governance for AI-Powered Applications                                      ┆ 35         │
│ 2  ┆ Securing the Modern SOC with NEC XON and Palo Alto Networks                    ┆ 15         │
└────┴────────────────────────────────────────────────────────────────────────────────┴────────────┘

📊 Sections by header level:
┌───────┬────────────┐
│ level ┆ n_sections │
╞═══════╪════════════╡
│ 1     ┆ 12         │
│ 2     ┆ 17         │
│ 3     ┆ 59        

### 🟰 Step 6.2: Semantic Summarization and Extraction



First, we define the structure of content we want LLM to extract from our content, which gets passed to the LLM as a pydantic model.

For this high level classification and summarization, we don't need the LLM to parse the whole PDF content.  We will pass only the Table of Contents.


In [ ]:
# 🎯 Define content categorization schema
class PDFContentCategorization(BaseModel):
    """AI-powered PDF content categorization."""
    summary: str = Field(description="Brief one sentence summary of the PDF given its table of contents")
    sections_about_model_training: List[str] = Field(description="List of headings that are specifically about model training")
    sections_about_soc_compliance: List[str] = Field(description="List of headings that are related to SOC compliance")
    sections_about_data_governance: List[str] = Field(description="List of headings that are related to data governance")
    products_mentioned: List[str] = Field(description="All product names mentioned in the PDF table of contents")

print("🎯 Content categorization schema defined")

# 🤖 AI-powered content analysis using table of contents
pdf_filtered_details = pdf_md_content_indexed.with_column(
    "content_categorization",
    fc.semantic.extract(fc.col("toc").cast(fc.StringType), PDFContentCategorization, model_alias="semantic_model")
).cache()

print("✅ AI content analysis complete!")

🎯 Content categorization schema defined
✅ AI content analysis complete!


### 📊 Step 6.3: Summarize the PDFs content



Let's see what insights AI extracted from our PDFs - summaries, products mentioned, and training-related sections.


In [ ]:
# 📊 Display whitepaper summaries and insights
print("="*70)
print("📄 WHITEPAPER ANALYSIS RESULTS")
print("="*70)


for row in pdf_filtered_details.to_pylist():
    print(f"\n📚 Whitepaper: {row['name']}")
    print(f"📝 Summary: {row['content_categorization']['summary']}")
    print(f"🏷️  Products mentioned: {row['content_categorization']['products_mentioned']}")
    print(f"🧠 Training sections: {row['content_categorization']['sections_about_model_training']}")
    print(f"🔒 SOC compliance sections: {row['content_categorization']['sections_about_soc_compliance']}")
    print(f"🔒 Data Governance sections: {row['content_categorization']['sections_about_data_governance']}")
    print("-" * 50)

📄 WHITEPAPER ANALYSIS RESULTS


Submitting requests for batch: 05f614de-e759-4ae0-a788-1d006427a8ea (model: gpt-4o-mini): 100%|██████████| 3/3 [00:00<00:00, 104.62req/s, estimated_input_tokens=1738, estimated_output_tokens=3072]
Awaiting responses for batch 05f614de-e759-4ae0-a788-1d006427a8ea (model: gpt-4o-mini): 100%|██████████| 3/3 [00:03<00:00,  1.32s/res]


📚 Whitepaper: Copy of PA_whitepaper_10-most-critical-security-risks-in-serverless-architectures-guide
📝 Summary: This document outlines the top 10 critical security risks associated with serverless architectures and offers insights into mitigation strategies.
🏷️  Products mentioned: ['Cortex Cloud']
🧠 Training sections: []
🔒 SOC compliance sections: []
🔒 Data Governance sections: []
--------------------------------------------------

📚 Whitepaper: AI Governance for AI-Powered Applications
📝 Summary: The document outlines a governance framework for AI-powered applications, detailing security implications, model training policies, and compliance considerations.
🏷️  Products mentioned: ['Cortex Cloud AI Security Posture Management', 'Palo Alto Networks']
🧠 Training sections: ['Model Training']
🔒 SOC compliance sections: ['Oversight of Relevant Compliance Frameworks', 'Ongoing Consideration of Current and Future Compliance Risk']
🔒 Data Governance sections: ['Discovery and Classification 

### 🔍 Step 6.4: Deep Dive into Specific Sections of Interest



Now we have PDF text content structured by section and we've annotated our table with classification data, we can examine sections of particular interest.

Imagine you're a data security engineer and you're only interested in Data Governence and SOC compliance.  We can cheaply filter content based on the markdown headings and only examine relevant parts of the whitepapers!


In [ ]:
# 🔍 Filter sections specifically about model training

soc_compliance_sections_df = pdf_filtered_details.explode("sections").filter(
    fc.col("sections").is_not_null() &
    fc.col("content_categorization").is_not_null() &
    (fc.array_contains(fc.col("content_categorization").sections_about_soc_compliance, fc.col("sections").heading) |
    fc.array_contains(fc.col("content_categorization").sections_about_data_governance, fc.col("sections").heading))

)


print("="*70)
print("🧠 SOC COMPLIANCE AND DATA GOVERNANCE DEEP DIVE")
print("="*70)
print(f"📊 Found {soc_compliance_sections_df.count()} sections about SOC compliance or Data Governance:")
print()

# Display training sections by document
for row in soc_compliance_sections_df.group_by("name").agg(fc.collect_list("sections").alias("sections")).to_pylist():
    print(f"📚 Document: {row['name']}")
    for section in row['sections']:
        print(f"  📖 Section: {section['heading']}")
        print(f"  📝 Content preview: {section['content'][:200]}...")
        print()
    print("-" * 50)

🧠 SOC COMPLIANCE AND DATA GOVERNANCE DEEP DIVE
📊 Found 3 sections about SOC compliance or Data Governance:

📚 Document: AI Governance for AI-Powered Applications
  📖 Section: Discovery and Classification Data Used for Al Model Training and Deployment
  📝 Content preview: Data is the lifeblood of Al models. It is therefore critical to have clear visibility into what data is being used across the Al life cycle. This includes data used for training models, for inference ...

  📖 Section: Policies to Prevent Data Poisoning
  📝 Content preview: Governance of training data is particularly important in the context of data poisoning attacks. If an adversary is able to manipulate the training data, it can introduce hidden backdoors or biases tha...

  📖 Section: Policies Regarding the Use of Sensitive Data for Training, Inference, and Fine-Tuning
  📝 Content preview: Organizations should establish clear policies governing how different types of data can be used for AI. The policies should be ba

### ✅ Step 6.5: Index the PDF Content by Topics of Interest

**What you'll do in this step:**

- Flatten sections to rows, add contact-info detectors (emails/URLs/phones) via UDFs.

- Create a small “topics” view from your LLM classification (model training, SOC compliance, data governance, products).

1) Use UDFs to read arrays from the dict/typed `content_categorization` column

In [ ]:
def _get_str_list(d: Any, key: str) -> List[str]:
    if isinstance(d, dict):
        v = d.get(key, [])
        return [str(x) for x in v] if isinstance(v, list) else []
    # Some fenic builds wrap this as a pydantic model; fall back via getattr
    try:
        v = getattr(d, key, [])
        return [str(x) for x in v] if isinstance(v, list) else []
    except Exception:
        return []

udf_get_products   = fc.udf(lambda d: _get_str_list(d, "products_mentioned"),
                            return_type=fc.ArrayType(fc.StringType))
udf_get_training   = fc.udf(lambda d: _get_str_list(d, "sections_about_model_training"),
                            return_type=fc.ArrayType(fc.StringType))
udf_get_soc        = fc.udf(lambda d: _get_str_list(d, "sections_about_soc_compliance"),
                            return_type=fc.ArrayType(fc.StringType))
udf_get_governance = fc.udf(lambda d: _get_str_list(d, "sections_about_data_governance"),
                            return_type=fc.ArrayType(fc.StringType))

2) Build topics DataFrame from the earlier LLM classification (`pdf_filtered_details.content_categorization`)

In [ ]:
topics_long = (
    pdf_filtered_details
      .select(
          "id",
          "name",
          udf_get_products(fc.col("content_categorization")).alias("products"),
          udf_get_training(fc.col("content_categorization")).alias("training"),
          udf_get_soc(fc.col("content_categorization")).alias("soc"),
          udf_get_governance(fc.col("content_categorization")).alias("governance"),
      )
      .cache()
)

3) Define a helper function to explode a topic list into (`doc_id`, name, topic, heading)

In [ ]:
def _explode_topic(df, col_name: str, topic_key: str):
    return (
        df
          .select("id", "name", fc.col(col_name).alias("headings"))
          .explode("headings")
          .filter(fc.col("headings").is_not_null())
          .select(
              fc.col("id").alias("doc_id"),
              "name",
              fc.lit(topic_key).alias("topic"),
              fc.col("headings").alias("heading"),
          )
    )

topics_training   = _explode_topic(topics_long, "training",   "training")
topics_soc        = _explode_topic(topics_long, "soc",        "soc")
topics_governance = _explode_topic(topics_long, "governance", "governance")

topics_union = topics_training.union(topics_soc).union(topics_governance)

4) Build a normalized section index from the “sections” column

In [ ]:
# (uses the array<struct{heading, level, content, parent_heading, full_path}> produced earlier)

sections_idx = (
    pdf_md_content_indexed
      .explode("sections")
      .filter(fc.col("sections").is_not_null())
      .select(
          fc.col("id").alias("doc_id"),
          "name",
          fc.col("sections").heading.alias("heading"),
          fc.col("sections").full_path.alias("full_path"),
          fc.col("sections").level.alias("level"),
          fc.col("sections").content.alias("content"),
      )
      .with_column("heading_norm", fc.text.lower(fc.coalesce(fc.col("heading"), fc.lit(""))))
      .cache()
)

topics_norm = topics_union.with_column(
    "heading_norm", fc.text.lower(fc.coalesce(fc.col("heading"), fc.lit("")))
)

In [ ]:
# Inner-join by (doc_id, normalized heading) to attach full section rows to topics
topic_sections = (
    topics_norm.join(
        sections_idx.select("doc_id","heading_norm","full_path","level","content"),
        on=["doc_id","heading_norm"],
        how="inner",
    )
)

5) Optional introspection into the number of sections per topic

In [ ]:
topic_counts = (
    topic_sections
      .group_by("topic")
      .agg(fc.count(fc.lit(1)).alias("n_sections"))
      .order_by("topic")
)

print("✅ Topic index ready (sections attached). Counts by topic:")
topic_counts.show()

✅ Topic index ready (sections attached). Counts by topic:
┌────────────┬────────────┐
│ topic      ┆ n_sections │
╞════════════╪════════════╡
│ governance ┆ 3          │
│ training   ┆ 1          │
└────────────┴────────────┘


6) Tiny helper to peek sections for a given topic (e.g., "training")

In [ ]:
def peek_topic(topic="training", limit=6):
    # collapse whitespace → extract a short snippet (first ~240 chars)
    one_line = fc.text.regexp_replace(
        fc.coalesce(fc.col("content"), fc.lit("")),
        fc.lit(r"\s+"), fc.lit(" ")
    )
    snippet = fc.text.regexp_substr(one_line, fc.lit(r"^.{0,240}"))

    (
        topic_sections
          .filter(fc.col("topic") == fc.lit(topic))
          .select("name", "topic", "full_path", "level", snippet.alias("snippet"))
          .order_by(["name","full_path","level"])
          .limit(limit)
          .show()
    )

# Example
peek_topic("training", limit=8)

┌──────────────────────────┬──────────┬──────────────────────────┬───────┬─────────────────────────┐
│ name                     ┆ topic    ┆ full_path                ┆ level ┆ snippet                 │
╞══════════════════════════╪══════════╪══════════════════════════╪═══════╪═════════════════════════╡
│ AI Governance for        ┆ training ┆ Establishing a           ┆ 3     ┆ Some large technology   │
│ AI-Powered Applications  ┆          ┆ Governance Framework for ┆       ┆ companies and research  │
│                          ┆          ┆ Al-Powered Applications  ┆       ┆ institutions are        │
│                          ┆          ┆ > Large Language Models  ┆       ┆ investing in training   │
│                          ┆          ┆ > Model Training         ┆       ┆ their own LLMs from     │
│                          ┆          ┆                          ┆       ┆ scratch. This is a      │
│                          ┆          ┆                          ┆       ┆ highly          

## **🛰️ Step 7. Expose an MCP server: “Whitepapers Catalog”**







Spin up a tiny **MCP** server that exposes typed and indexed access to your curated whitepaper dataset as tools. This lets any MCP-capable host or agents query clean tables and indexes to get fast, bounded, and auditable answers, not messy PDFs.


### **What this cell does**

* **Normalizes & embeds** your parsed sections into a `whitepaper_sections` table (idempotent).

* Saves **topics** into `whitepaper_topics` (idempotent).

* **Registers six tools** in the catalog:

  * `list_whitepapers()`: whitepaper names \+ section counts

  * `sections_by_topic(topic)`: topic-aligned sections (e.g., *training*, *soc*, *governance*)

  * `search_sections(query)`: case-insensitive substring on heading/content

* Starts a lightweight **HTTP MCP** server and prints the endpoint (random free port).

**Why this design?**

- **1) Deterministic structure.** Tools return **tabular rows** with stable columns and typed parameters, so hosts don’t have to infer structure from raw text. That makes chaining, filtering, and UI rendering predictable.  

- **2) Faster, cheaper retrieval.** We embed once and **reuse** vectors. Queries run as push-down filters (substring/joins/similarity) on the table. It's much lighter than repeatedly re-parsing large PDFs.  

- **3) Safer contracts.** Tool params (with allowed values/defaults) constrain queries and help avoid brittle prompts or excessive payloads; **result limits** keep responses bounded.  

- **4) Composability.** Because the data is in tables, you can add new views (e.g., “governance only”, “contact-bearing sections”), join across indexes, or change scoring without touching downstream agents.  

- **5) Better observability.** With explicit tables and views, you can log/query coverage, tool hit-rates, and section-level provenance. This is impossible to do reliably when an agent free-reads PDFs.

In [ ]:
import asyncio, json, re, operator, math
from functools import reduce
from pprint import pprint

from fastmcp import Client

from fenic.api.mcp.server import create_mcp_server, run_mcp_server_async
from fenic.core.mcp.types import ToolParam

# Prereqs in runtime:
# - pdf_filtered_details  (from Step 6.3)
# - topics_union          (from Step 6.5)
# - session               (already created)

# ------------------------------
# 1) Simple regex UDFs for contact-like signals
# ------------------------------

EMAIL_RE = re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}")
URL_RE   = re.compile(r"(https?://[^\s)]+)")
PHONE_RE = re.compile(r"(?:\+?\d{1,3}[-.\s]?)?(?:\(?\d{2,4}\)?[-.\s]?)?\d{3,4}[-.\s]?\d{3,4}")

def _extract_emails(text_s: str):
    if not text_s: return []
    return list({m.group(0) for m in EMAIL_RE.finditer(text_s)})

def _extract_urls(text_s: str):
    if not text_s: return []
    return list({m.group(0) for m in URL_RE.finditer(text_s)})

def _extract_phones(text_s: str):
    if not text_s: return []
    vals = [m.group(0) for m in PHONE_RE.finditer(text_s)]
    import re as _re
    return [v.strip() for v in vals if 7 <= len(_re.sub(r"\D","",v)) <= 15]

udf_emails = fc.udf(_extract_emails, return_type=fc.ArrayType(fc.StringType))
udf_urls   = fc.udf(_extract_urls,   return_type=fc.ArrayType(fc.StringType))
udf_phones = fc.udf(_extract_phones, return_type=fc.ArrayType(fc.StringType))
udf_len    = fc.udf(lambda arr: 0 if arr is None else len(arr), return_type=fc.IntegerType)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# ------------------------------
# 2) Normalize section rows and precompute vectors
# ------------------------------
sections_long = (
    pdf_filtered_details
      .explode("sections")
      .filter(fc.col("sections").is_not_null())
      .select(
          "id","name",
          fc.col("sections").heading.alias("heading"),
          fc.col("sections").level.alias("level"),
          fc.col("sections").content.alias("content"),
          fc.col("sections").full_path.alias("full_path"),
      )
      # contact signals on raw content
      .with_column("emails", udf_emails(fc.col("content")))
      .with_column("urls",   udf_urls(fc.col("content")))
      .with_column("phones", udf_phones(fc.col("content")))
      .with_column(
          "has_contact_info",
          (udf_len(fc.col("emails")) > 0)
          | (udf_len(fc.col("urls"))   > 0)
          | (udf_len(fc.col("phones")) > 0)
      )
      # joined text for semantic tasks (optionally strip page-break markers)
      .with_column(
          "joined_text",
          text.concat(
              fc.coalesce(fc.col("heading"), fc.lit("")),
              fc.lit("\n\n"),
              # If you want to strip literal PAGE BREAK markers, uncomment next line and replace fc.col("content")
              # text.replace(fc.coalesce(fc.col("content"), fc.lit("")), fc.lit("--- PAGE BREAK ---"), fc.lit(" "))
              fc.coalesce(fc.col("content"), fc.lit(""))
          )
      )
      # precompute section vectors
      .with_column("joined_vec", semantic.embed(fc.col("joined_text")))
      .cache()
)

In [ ]:
# ------------------------------
# 3) Persist tables (idempotent)
# ------------------------------
topics_union.write.save_as_table("whitepaper_topics", mode="overwrite")
sections_long.write.save_as_table("whitepaper_sections", mode="overwrite")

topics_tbl   = session.table("whitepaper_topics")
sections_tbl = session.table("whitepaper_sections")

DEBUG:fenic._backends.local.catalog:Beginning DuckDB transaction
DEBUG:fenic._backends.local.catalog:Committing DuckDB transaction
Submitting requests for batch: a89f94be-9d6a-4af9-b0d3-27faecd0211d (model: text-embedding-3-large):   0%|          | 0/88 [00:00<?, ?req/s]/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
Submitting requests for batch: a89f94be-9d6a-4af9-b0d3-27faecd0211d (model: text-embedding-3-large): 100%|██████████| 88/88 [00:00<00:00, 93.16req/s, estimated_input_tokens=38409, estimated_output_tokens=0]
Awaiting responses for batch a89f94be-9d6a-4af9-b0d3-27faecd0211d (model: text-embedding-3-large): 100%|██████████| 88/88 [00:05<00:00, 17.28res/s]
DEBUG:fenic._backends.local.catalog:Beginning DuckDB 

#### 🛠️ **Tool reference (quick schema)**

| Tool | Params | Returns | Notes |
| ----- | ----- | ----- | ----- |
| `list_whitepapers` | — | `id, name, n_sections` | One row per whitepaper |
| `sections_by_topic` | `topic: str ∈ {training, soc, governance}` | `doc_id, name, topic, heading, content, full_path, level` | Topic match is case-insensitive |
| `search_sections` | `query: str` | `id, name, heading, full_path, level` | `ILIKE` over heading/content |

In [ ]:
# ------------------------------
# 4) Register MCP tools
# ------------------------------

# Tool 1: list_whitepapers()
wp_counts = (
    sections_tbl
      .group_by("id","name")
      .agg(fc.count(fc.lit(1)).alias("n_sections"))
      .order_by("name")
)
session.catalog.create_tool(
    tool_name="list_whitepapers",
    tool_description="List whitepapers with total section counts.",
    tool_query=wp_counts,
    tool_params=[],
)

# Tool 2: sections_by_topic(topic) — case-insensitive topic match
topic_param = fc.tool_param("topic", fc.StringType)

topics_left = (
    topics_tbl
      .select("doc_id", "heading", "topic")
      .with_column("topic_norm", fc.text.lower(fc.coalesce(fc.col("topic"), fc.lit(""))))
)

sections_right = (
    sections_tbl
      .select("id", "name", "heading", "full_path", "content", "level")
      .with_column("doc_id_right", fc.col("id"))
      .drop("id")
)

sections_by_topic = (
    topics_left
      .join(
          sections_right,
          left_on=["doc_id", "heading"],
          right_on=["doc_id_right", "heading"],
          how="inner",
      )
      .filter(fc.col("topic_norm") == fc.text.lower(topic_param))
      .select(
          fc.col("doc_id"),
          fc.col("name"),
          fc.col("topic_norm").alias("topic"),
          fc.col("heading"),
          fc.col("content"),
          fc.col("full_path"),
          fc.col("level"),
      )
      .order_by(["name", "full_path", "level"])
)

session.catalog.create_tool(
    tool_name="sections_by_topic",
    tool_description="Return sections for a given topic (case-insensitive: training|soc|governance).",
    tool_query=sections_by_topic,
    tool_params=[ToolParam(
        name="topic",
        description="Topic key (case-insensitive)",
        allowed_values=["training", "soc", "governance"]
    )],
    result_limit=500,
)


# Tool 3: search_sections(query: str) — case-insensitive substring
query_param = fc.tool_param("query", fc.StringType)
pattern = text.concat(fc.lit("%"), query_param, fc.lit("%"))

search_q = (
    sections_tbl
      .filter(
          (fc.col("content").ilike(pattern)) |
          (fc.col("heading").ilike(pattern))
      )
      .select("id","name","heading","full_path","level")
      .order_by(["name","full_path","level"])
)

session.catalog.create_tool(
    tool_name="search_sections",
    tool_description="Substring search over section heading/content (case-insensitive).",
    tool_query=search_q,
    tool_params=[ToolParam(name="query", description="Substring to search for (case-insensitive)")],
    result_limit=200,
)

True

In [ ]:
# ------------------------------
# 5) Start/refresh MCP server (HTTP, stateless)
# ------------------------------
def _find_free_port(host="127.0.0.1"):
    import socket
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.bind((host,0))
    _, p = s.getsockname()
    s.close()
    return p

# Grab the catalog tools you just registered above
catalog_tools = session.catalog.list_tools()

# Build the server
server = create_mcp_server(
    session, server_name="Fenic MCP – Whitepapers",
    user_defined_tools=catalog_tools
)

HOST, PORT = "127.0.0.1", _find_free_port()

# If re-running, cancel the old task safely
try:
    mcp_task.cancel()
except Exception:
    pass

mcp_task = asyncio.create_task(
    run_mcp_server_async(
        server,
        transport="http",
        host=HOST,
        port=PORT,
        stateless_http=True,
        path="/mcp",
        log_level="warning",
    )
)
await asyncio.sleep(0.6)

print(f"✅ MCP HTTP server ready at http://{HOST}:{PORT}/mcp")
print("   Tools:", [t.name for t in session.catalog.list_tools()])

✅ MCP HTTP server ready at http://127.0.0.1:50755/mcp
   Tools: ['list_whitepapers', 'sections_by_topic', 'search_sections']


## **🧪 Smoke test your MCP tools**





This cell does a **live end-to-end** probe against the MCP server from Step 7.

### **What this cell does**

* Connects to the Step 7 MCP endpoint at `http://{HOST}:{PORT}/mcp`.

* Lists tools and **executes representative calls**:

  1. `list_whitepapers()`

  2. `sections_by_topic(topic in {training,soc,governance})`

  4. `search_sections("latency"|"privacy"|"training")`
* **Normalizes outputs** from different MCP result shapes and prints tables with a sensible column order and row cap.


In [ ]:
BASE_URL = f"http://{HOST}:{PORT}/mcp"  # printed in Step 7
MAX_ROWS = 15

def _md_table(rows):
    if not rows:
        return print("(no rows)")

    # If rows is not a list, just print it
    if not isinstance(rows, list):
        print(rows)
        return

    # If it's a list of strings, print as list
    if rows and isinstance(rows[0], str):
        print(rows)
        return

    # normalize list[list] -> list[dict]
    if isinstance(rows[0], list):
        rows = [{str(i): v for i, v in enumerate(r)} for r in rows]

    # If it's list[dict], print table
    if rows and isinstance(rows[0], dict):
        cols = list(rows[0].keys())
        print("| " + " | ".join(cols) + " |")
        print("| " + " | ".join(["---"]*len(cols)) + " |")
        for r in rows[:MAX_ROWS]:
            print("| " + " | ".join(str(r.get(c, "")) for c in cols) + " |")
    else:
        print(rows)

async def demo():
    async with Client(BASE_URL) as c:
        print("\n# list_whitepapers")
        r1 = await c.call_tool("list_whitepapers")
        # Try to extract rows from likely locations
        data = getattr(r1, "data", None)
        rows = getattr(data, "rows", None) if data else None

        _md_table(rows or [])

        print("\n# sections_by_topic('training')")
        r2 = await c.call_tool("sections_by_topic", {"topic": "training"})
        data = getattr(r2, "data", None)
        rows = getattr(data, "rows", None) if data else None
        if rows is None and hasattr(r2, "content") and r2.content:
            rows = r2.content[0].text
        _md_table(rows or [])

        print("\n# search_sections('latency')")
        r3 = await c.call_tool("search_sections", {"query": "latency"})
        data = getattr(r3, "data", None)
        rows = getattr(data, "rows", None) if data else None
        if rows is None and hasattr(r3, "content") and r3.content:
            rows = r3.content[0].text
        _md_table(rows or [])


await demo()

INFO:fenic._backends.local.execution:Execution ID: c1db4d07-5f3d-42ae-bb8c-53d6b193d691
INFO:fenic.core.mcp._server:Completed query for list_whitepapers
INFO:fenic.core.mcp._server:Query executed in 25.49ms, returned 3 rows, language model cost: $0.000000, embedding model cost: $0.000000
INFO:fenic._backends.local.execution:Execution ID: ba29963c-cfb1-4093-a45e-6b3ad25d2e56
INFO:fenic.core.mcp._server:Completed query for sections_by_topic
INFO:fenic.core.mcp._server:Query executed in 17.91ms, returned 1 rows, language model cost: $0.000000, embedding model cost: $0.000000
INFO:fenic._backends.local.execution:Execution ID: ef5cccfa-8735-4eb7-854b-4737cad19295
INFO:fenic.core.mcp._server:Completed query for search_sections
INFO:fenic.core.mcp._server:Query executed in 15.69ms, returned 2 rows, language model cost: $0.000000, embedding model cost: $0.000000



# list_whitepapers
| id | name | n_sections |
| --- | --- | --- |
| 0 | AI Governance for AI-Powered Applications | 35 |
| 3 | Copy of PA_whitepaper_10-most-critical-security-risks-in-serverless-architectures-guide | 38 |
| 2 | Securing the Modern SOC with NEC XON and Palo Alto Networks | 15 |

# sections_by_topic('training')
| doc_id | name | topic | heading | content | full_path | level |
| --- | --- | --- | --- | --- | --- | --- |
| 0 | AI Governance for AI-Powered Applications | training | Model Training | Some large technology companies and research institutions are investing in training their own LLMs from scratch. This is a highly resource-intensive process that requires massive compute power and datasets. It does, however, allow companies to have full control over the model architecture, training data, and optimization process, as well as to maintain full intellectual property rights over the resulting models. | Establishing a Governance Framework for Al-Powered Applications >

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


### 🔚 Optional teardown (uncomment if you want to stop the MCP task)

In [ ]:
try:
    mcp_task.cancel()
    print("🛑 MCP server task cancelled.")
except Exception as e:
    print("Teardown note:", e)

🛑 MCP server task cancelled.


## ➡️ Next steps: Test two more search tools

If you want an agent or your MCP client to query the server with questions and retrieve content from your whitepapers, here are two tools you could add to your catalog:

  * `qa_sections(question, min_sim=0.35)`: **vector search** using `joined_vec` \+ cosine similarity.

  * `qa_sections_kw(kw1..kw6)`: **keyword OR** search (up to 6 tokens) with a simple hit score.



🛠️ **Two more tool references:**

| Tool | Params | Returns | Notes |
| ----- | ----- | ----- | ----- |
| `qa_sections` | `question: str`, `min_sim: float = 0.35` | `id, name, full_path, level, heading, content, sim` | Cosine similarity via `embedding.compute_similarity` |
| `qa_sections_kw` | `kw1..kw6: str = ""` | `id, name, full_path, level, heading, content, kw_score` | OR-match; rows scored by token hits |

In [ ]:
# Tool: qa_sections(question: str, min_sim: float = 0.35)
question_param = fc.tool_param("question", fc.StringType)
min_sim_param  = fc.tool_param("min_sim",  fc.FloatType)

q_vec = semantic.embed(question_param)

qa_sections_sim = (
    sections_tbl
      .with_column("sim", embedding.compute_similarity(fc.col("joined_vec"), q_vec))
      .filter(fc.col("sim") >= min_sim_param)
      .select("id","name","full_path","level","heading","content","sim")
      .order_by(fc.col("sim").desc())
)

session.catalog.create_tool(
    tool_name="qa_sections",
    tool_description="Similarity-ranked sections that best answer the natural-language question.",
    tool_query=qa_sections_sim,
    tool_params=[
        ToolParam(name="question", description="Your natural-language question"),
        ToolParam(name="min_sim", description="Minimum cosine similarity", has_default=True, default_value=0.35),
    ],
    result_limit=40,
)

# Tool: qa_sections_kw(keyword OR, up to 6 tokens)
kw_params = [fc.tool_param(f"kw{i}", fc.StringType) for i in range(1, 7)]
patterns  = [text.concat(fc.lit("%"), p, fc.lit("%")) for p in kw_params]

conds = [
    (raw.is_not_null() & (raw != fc.lit(""))) &
    (fc.col("content").ilike(pat) | fc.col("heading").ilike(pat))
    for pat, raw in zip(patterns, kw_params)
]

any_cond = reduce(operator.or_, conds)
score = None
for c in conds:
    term_score = fc.when(c, fc.lit(1)).otherwise(fc.lit(0))
    score = term_score if score is None else (score + term_score)

qa_sections_kw = (
    sections_tbl
      .with_column("kw_score", score)
      .filter(any_cond)
      .select("id","name","full_path","level","heading","content","kw_score")
      .order_by([fc.col("kw_score").desc(), "name","full_path","level"])
)
session.catalog.create_tool(
    tool_name="qa_sections_kw",
    tool_description="Keyword QA: OR-match up to 6 tokens; returns rows scored by token hits.",
    tool_query=qa_sections_kw,
    tool_params=[
        ToolParam(name="kw1", description="keyword 1", has_default=True, default_value=""),
        ToolParam(name="kw2", description="keyword 2", has_default=True, default_value=""),
        ToolParam(name="kw3", description="keyword 3", has_default=True, default_value=""),
        ToolParam(name="kw4", description="keyword 4", has_default=True, default_value=""),
        ToolParam(name="kw5", description="keyword 5", has_default=True, default_value=""),
        ToolParam(name="kw6", description="keyword 6", has_default=True, default_value=""),
    ],
    result_limit=500,
)

## 🎉 What just happened?

**You just witnessed the future of document processing:**

1. **📄 PDF → Markdown**: AI converted complex PDFs into clean, structured markdown while preserving formatting and hierarchy
2. 🕋 **Chunking and Deduplication** - Use fenic's text manipulation and fuzzy matching algorithms to dedup structured data
3. **🧠 Content Analysis**: AI analyzed document structure and extracted key insights like products mentioned and training sections  
4. **📊 Structured Data**: Transformed unstructured PDFs into queryable, structured data
5. **🔍 Smart Filtering**: Automatically identified and extracted only relevant sections
6. **🛰️ MCP Server:** MCP server that exposes your curated whitepaper dataset as tools for MCP clients and agents

**This is semantic AI in action** - understanding document content, not just extracting text. Perfect for research analysis, document management, and content discovery.

**Try this with your own PDFs** - just change the huggingface directory or use your own s3 bucket and watch fenic work it's magic!

> ➡️ **SEE ALSO:** [fenic + Hugging Face Datasets: One-Line Agent Context Hydration](https://www.typedef.ai/blog/fenic-hugging-face-datasets-one-line-agent-context-hydration)

In [ ]:
# 🧹 Cleanup
print("🧹 Cleaning up downloaded files...")
shutil.rmtree(DATA_DIR)
session.stop()
print("✅ Cleanup complete!")

INFO:fenic._backends.local.async_utils:Event loop shutdown complete


🧹 Cleaning up downloaded files...

Session Usage Summary:
  App Name: pdf_process_and_dedup_demo2
  Session ID: 63801f59-51b6-400b-abb4-b2c8176eeac1
  Total queries executed: 64
  Total execution time: 44554.26ms
  Total rows processed: 623
  Total language model cost: $0.530542
  Total language model requests: 28
  Total language model tokens: 24,991 input tokens, 0 cached input tokens, 50,418 output tokens
  Total embedding model cost: $0.004993
  Total embedding model requests: 88
  Total embedding model tokens: 38,409 input tokens
  Total cost: $0.535535
✅ Cleanup complete!
